In [11]:
import pandas as pd 



In [12]:
data = pd.read_excel('./data/Wind_Turbine_Database_en.xlsx')
print(data.head())

  Province_Territory          Project Name  Total Project Capacity (MW)  \
0            Alberta  Ardenville Wind Farm                         69.0   
1            Alberta  Ardenville Wind Farm                         69.0   
2            Alberta  Ardenville Wind Farm                         69.0   
3            Alberta  Ardenville Wind Farm                         69.0   
4            Alberta  Ardenville Wind Farm                         69.0   

  Turbine Identifier  Turbine Number  Number of Turbines in Project  \
0               AWF1               1                             23   
1               AWF2               2                             23   
2               AWF3               3                             23   
3               AWF4               4                             23   
4               AWF5               5                             23   

  Turbine Rated Capacity (kW)  Rotor Diameter (m) Hub Height (m) Manufacturer  \
0                        3000            

In [13]:
# Filter for Ontario province
ontario_data = data[data['Province_Territory'] == 'Ontario']

print(f"Total records: {len(data)}")
print(f"Ontario records: {len(ontario_data)}")
print(ontario_data.head())

Total records: 7841
Ontario records: 2712
     Province_Territory                 Project Name  \
2736            Ontario  Adelaide Wind Energy Centre   
2737            Ontario  Adelaide Wind Energy Centre   
2738            Ontario  Adelaide Wind Energy Centre   
2739            Ontario  Adelaide Wind Energy Centre   
2740            Ontario  Adelaide Wind Energy Centre   

      Total Project Capacity (MW) Turbine Identifier  Turbine Number  \
2736                        59.94               AWE1               1   
2737                        59.94               AWE2               2   
2738                        59.94               AWE3               3   
2739                        59.94               AWE4               4   
2740                        59.94               AWE5               5   

      Number of Turbines in Project Turbine Rated Capacity (kW)  \
2736                             37                        1620   
2737                             37                   

In [14]:
import os
import geopandas as gpd
from shapely.geometry import Point

# Extract year helper function
def extract_year(val):
    """Extract year from commissioning value (handles ranges like '2006/2008')"""
    if pd.isna(val) or str(val) == 'nan':
        return None
    try:
        val_str = str(val).strip()
        if '/' in val_str:
            val_str = val_str.split('/')[0]  # Take first year from range
        return int(float(val_str))
    except:
        return None

# Add extracted year column
ontario_data['Year'] = ontario_data['Commissioning'].apply(extract_year)

# Create GeoDataFrame
geometry = [Point(xy) for xy in zip(ontario_data['Longitude'], ontario_data['Latitude'])]
turbines_gdf = gpd.GeoDataFrame(ontario_data, geometry=geometry, crs='EPSG:4326')

# Save all columns as GeoJSON for backend to serve
output_path = 'results/turbines.geojson'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
turbines_gdf.to_file(output_path, driver='GeoJSON')

print(f"✓ Saved {len(turbines_gdf)} Ontario turbines → {output_path}")
print(f"  Years: {turbines_gdf['Year'].min():.0f} - {turbines_gdf['Year'].max():.0f}")
print("\nStyling is now controlled in frontend/index.html")


✓ Saved 2712 Ontario turbines → results/turbines.geojson
  Years: 1995 - 2021

Styling is now controlled in frontend/index.html
